## SRP080993 / GSE85239

**paper:** [PMID: 28821769](https://pmc.ncbi.nlm.nih.gov/articles/PMC5562767/) - Patterns of transcriptional parallelism and variation in the developing olfactory system of Drosophila species, 2017

**date, curator:** 2026-07-15, Sara Carsanaro

**notes**
* D. melanogaster is not included in this dataset, see GSE75986
* all strains are from methods section
* per methods, all adults are 7-10 days post eclosion
* per methods - adult samples are mixed sex, others are unknown sex

### annotation summary

In [21]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Antenna,UBERON:0000972,antenna,perfect match
2,Antennal disc,UBERON:6001767,insect antennal disc,perfect match


In [22]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,48hrs after pupal formation,UBERON:0000070,pupal stage,perfect match
2,10hrs after pupal formation,UBERON:0000070,pupal stage,perfect match
4,3rd instar larvae,UBERON:8000002,third instar larva stage,perfect match
6,Adult,UBERON:0000066,fully formed stage,perfect match
8,40hrs after pupal formation,UBERON:0000070,pupal stage,perfect match
10,8hrs after pupal formation,UBERON:0000070,pupal stage,perfect match


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP080993"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

## validation
path_to_v_script = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/validate_annotations.py'
path_to_rules = '/Users/scarsana/Desktop/git/continuous_integration/validate_annotations/rules/'
val_output = "{}{}/validation.tsv".format(path_to_output_main, experiment_id)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
Be patient, it may take a few minutes.
0it [00:00, ?it/s]
0 samples dont have attributes, try to find them somewhere else
0it [00:00, ?it/s]
0 samples dont have attributes


### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX2004106,SRP080993,Illumina HiSeq 2500,SRS1603673,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262779,Antenna,48hrs after pupal formation,,,,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p40_2,"SAMN05513848,GSM2262779",48,hour post pupal formation,,,,,,,,,15/07/2026,"Dissected tissue was stored in Trizol reagent for pooling. RNA was extracted using the RNeasy kit (Qiagen) according to the manufacturer’s instructions, including an on-column DNase digestion (Qiagen). RNA concentrations were measured and 700ng RNA was diluted to 60ul total volume with H2O. RNA sequencing libraries were prepared with TruSeq Stranded mRNA Sample Prep Kit (Illumina) according to manufacturer’s instructions. For the RNA fragmentation step, 94˚C, 2min was used with the intention to obtain a median size ~185bp. PCR amplification was done with 15 cycles. A total of 24 multiplexed libraries (barcoded) were accessed for quality and combined together before separation into two identical pooled libraries, which were then subjected to cluster generation followed by Illumina 50bp paired-end sequencing by the UNC High-Throughput Sequencing Facility (HTSF).",,,,48hr APF pupal antenna,70,48hrs after pupal formation,,TRANSCRIPTOMIC,cDNA
1,SRX2004105,SRP080993,Illumina HiSeq 2500,SRS1603670,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262778,Antenna,48hrs after pupal formation,,,,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p40_1,"SAMN05513849,GSM2262778",48,hour post pupal formation,,,,,,,,,15/07/2026,"Dissected tissue was stored in Trizol reagent for pooling. RNA was extracted using the RNeasy kit (Qiagen) according to the manufacturer’s instructions, including an on-column DNase digestion (Qiagen). RNA concentrations were measured and 700ng RNA was diluted to 60ul total volume with H2O. RNA sequencing libraries were prepared with TruSeq Stranded mRNA Sample Prep Kit (Illumina) according to manufacturer’s instructions. For the RNA fragmentation step, 94˚C, 2min was used with the intention to obtain a median size ~185bp. PCR amplification was done with 15 cycles. A total of 24 multiplexed libraries (barcoded) were accessed for quality and combined together before separation into two identical pooled libraries, which were then subjected to cluster generation followed by Illumina 50bp paired-end sequencing by the UNC High-Throughput Sequencing Facility (HTSF).",,,,48hr APF pupal antenna,70,48hrs after pupal formation,,TRANSCRIPTOMIC,cDNA
2,SRX2004104,SRP080993,Illumina HiSeq 2500,SRS1603668,,,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262777,Antennal disc,10hrs after pupal formation,,,,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p8_2,"SAMN05513850,GSM2262777",10,hour post pupal formation,,,,,,,,,15/07/2026,"Dissected tissue was stored in Trizol reagent for pooling. RNA was extracted using the RNeasy kit (Qiagen) according to the manufacturer’s instructions, including an on-column DNase digestion (Qiagen). RNA concentrations were measured and 700ng RNA was diluted to 60ul total volume with H2O. RNA sequencing libraries were prepared with TruSeq Stranded mRNA Sample Prep Kit (Illumina) according to manufacturer’s instructions. For the RNA fragmentation step, 94˚C, 2min was used with the intention to obtain a median size ~185bp. PCR amplification was done with 15 cycles. A total of 24 multiplexed libraries (barcoded) 

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Antenna' 'Antennal disc']


In [6]:

# Antenna
library.loc[library["infoOrgan"] == "Antenna", "anatId"] = "UBERON:0000972"
library.loc[library["infoOrgan"] == "Antenna", "anatName"] = "antenna"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Antenna", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Antenna", "anatBiologicalStatus"] = "not documented"

# Antennal disc
library.loc[library["infoOrgan"] == "Antennal disc", "anatId"] = "UBERON:6001767"
library.loc[library["infoOrgan"] == "Antennal disc", "anatName"] = "insect antennal disc"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Antennal disc", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Antennal disc", "anatBiologicalStatus"] = "not documented"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX2004106,SRP080993,Illumina HiSeq 2500,SRS1603673,UBERON:0000972,antenna,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262779,Antenna,48hrs after pupal formation,perfect match,not documented,,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p40_2,"SAMN05513848,GSM2262779",48,hour post pupal formation,,,,,,,,,15/07/2026,"Dissected tissue was stored in Trizol reagent for pooling. RNA was extracted using the RNeasy kit (Qiagen) according to the manufacturer’s instructions, including an on-column DNase digestion (Qiagen). RNA concentrations were measured and 700ng RNA was diluted to 60ul total volume with H2O. RNA sequencing libraries were prepared with TruSeq Stranded mRNA Sample Prep Kit (Illumina) according to manufacturer’s instructions. For the RNA fragmentation step, 94˚C, 2min was used with the intention to obtain a median size ~185bp. PCR amplification was done with 15 cycles. A total of 24 multiplexed libraries (barcoded) were accessed for quality and combined together before separation into two identical pooled libraries, which were then subjected to cluster generation followed by Illumina 50bp paired-end sequencing by the UNC High-Throughput Sequencing Facility (HTSF).",,,,48hr APF pupal antenna,70,48hrs after pupal formation,,TRANSCRIPTOMIC,cDNA
1,SRX2004105,SRP080993,Illumina HiSeq 2500,SRS1603670,UBERON:0000972,antenna,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262778,Antenna,48hrs after pupal formation,perfect match,not documented,,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p40_1,"SAMN05513849,GSM2262778",48,hour post pupal formation,,,,,,,,,15/07/2026,"Dissected tissue was stored in Trizol reagent for pooling. RNA was extracted using the RNeasy kit (Qiagen) according to the manufacturer’s instructions, including an on-column DNase digestion (Qiagen). RNA concentrations were measured and 700ng RNA was diluted to 60ul total volume with H2O. RNA sequencing libraries were prepared with TruSeq Stranded mRNA Sample Prep Kit (Illumina) according to manufacturer’s instructions. For the RNA fragmentation step, 94˚C, 2min was used with the intention to obtain a median size ~185bp. PCR amplification was done with 15 cycles. A total of 24 multiplexed libraries (barcoded) were accessed for quality and combined together before separation into two identical pooled libraries, which were then subjected to cluster generation followed by Illumina 50bp paired-end sequencing by the UNC High-Throughput Sequencing Facility (HTSF).",,,,48hr APF pupal antenna,70,48hrs after pupal formation,,TRANSCRIPTOMIC,cDNA
2,SRX2004104,SRP080993,Illumina HiSeq 2500,SRS1603668,UBERON:6001767,insect antennal disc,,,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262777,Antennal disc,10hrs after pupal formation,perfect match,not documented,,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p8_2,"SAMN05513850,GSM2262777",10,hour post pupal formation,,,,,,,,,15/07/2026,"Dissected tissue was stored in Trizol reagent for pooling. RNA was extracted using the RNeasy kit (Qiagen) according to the manufacturer’s instructions, including an on-column DNase digestion (Qiagen). RNA concentrations were measured and 700ng RNA was diluted to 60ul total volume with H2O. RNA sequencing libraries were prepared with TruSeq Stranded mRNA Sample Prep Kit (Illumina) according to manufacturer’s instructions. For the RNA fragmentation step, 94˚C

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['10hrs after pupal formation' '3rd instar larvae'
 '40hrs after pupal formation' '48hrs after pupal formation'
 '8hrs after pupal formation' 'Adult']


In [8]:
# Adult
library.loc[library["infoStage"] == "Adult", "stageId"] = "UBERON:0000066"
library.loc[library["infoStage"] == "Adult", "stageName"] = "fully formed stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "Adult", "stageAnnotationStatus"] = "perfect match"

# 3rd instar larvae
library.loc[library["infoStage"] == "3rd instar larvae", "stageId"] = "UBERON:8000002"
library.loc[library["infoStage"] == "3rd instar larvae", "stageName"] = "third instar larva stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "3rd instar larvae", "stageAnnotationStatus"] = "perfect match"

# 10hrs after pupal formation
library.loc[library["infoStage"] == "10hrs after pupal formation", "stageId"] = "UBERON:0000070"
library.loc[library["infoStage"] == "10hrs after pupal formation", "stageName"] = "pupal stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "10hrs after pupal formation", "stageAnnotationStatus"] = "perfect match"

# 40hrs after pupal formation
library.loc[library["infoStage"] == "40hrs after pupal formation", "stageId"] = "UBERON:0000070"
library.loc[library["infoStage"] == "40hrs after pupal formation", "stageName"] = "pupal stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "40hrs after pupal formation", "stageAnnotationStatus"] = "perfect match"

# 48hrs after pupal formation
library.loc[library["infoStage"] == "48hrs after pupal formation", "stageId"] = "UBERON:0000070"
library.loc[library["infoStage"] == "48hrs after pupal formation", "stageName"] = "pupal stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "48hrs after pupal formation", "stageAnnotationStatus"] = "perfect match"

# 8hrs after pupal formation
library.loc[library["infoStage"] == "8hrs after pupal formation", "stageId"] = "UBERON:0000070"
library.loc[library["infoStage"] == "8hrs after pupal formation", "stageName"] = "pupal stage"
# perfect match, missing child term, other
library.loc[library["infoStage"] == "8hrs after pupal formation", "stageAnnotationStatus"] = "perfect match"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX2004106,SRP080993,Illumina HiSeq 2500,SRS1603673,UBERON:0000972,antenna,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262779,Antenna,48hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p40_2,"SAMN05513848,GSM2262779",48,hour post pupal formation,,,,,,,,,15/07/2026,"Dissected tissue was stored in Trizol reagent for pooling. RNA was extracted using the RNeasy kit (Qiagen) according to the manufacturer’s instructions, including an on-column DNase digestion (Qiagen). RNA concentrations were measured and 700ng RNA was diluted to 60ul total volume with H2O. RNA sequencing libraries were prepared with TruSeq Stranded mRNA Sample Prep Kit (Illumina) according to manufacturer’s instructions. For the RNA fragmentation step, 94˚C, 2min was used with the intention to obtain a median size ~185bp. PCR amplification was done with 15 cycles. A total of 24 multiplexed libraries (barcoded) were accessed for quality and combined together before separation into two identical pooled libraries, which were then subjected to cluster generation followed by Illumina 50bp paired-end sequencing by the UNC High-Throughput Sequencing Facility (HTSF).",,,,48hr APF pupal antenna,70,48hrs after pupal formation,,TRANSCRIPTOMIC,cDNA
1,SRX2004105,SRP080993,Illumina HiSeq 2500,SRS1603670,UBERON:0000972,antenna,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262778,Antenna,48hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p40_1,"SAMN05513849,GSM2262778",48,hour post pupal formation,,,,,,,,,15/07/2026,"Dissected tissue was stored in Trizol reagent for pooling. RNA was extracted using the RNeasy kit (Qiagen) according to the manufacturer’s instructions, including an on-column DNase digestion (Qiagen). RNA concentrations were measured and 700ng RNA was diluted to 60ul total volume with H2O. RNA sequencing libraries were prepared with TruSeq Stranded mRNA Sample Prep Kit (Illumina) according to manufacturer’s instructions. For the RNA fragmentation step, 94˚C, 2min was used with the intention to obtain a median size ~185bp. PCR amplification was done with 15 cycles. A total of 24 multiplexed libraries (barcoded) were accessed for quality and combined together before separation into two identical pooled libraries, which were then subjected to cluster generation followed by Illumina 50bp paired-end sequencing by the UNC High-Throughput Sequencing Facility (HTSF).",,,,48hr APF pupal antenna,70,48hrs after pupal formation,,TRANSCRIPTOMIC,cDNA
2,SRX2004104,SRP080993,Illumina HiSeq 2500,SRS1603668,UBERON:6001767,insect antennal disc,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262777,Antennal disc,10hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p8_2,"SAMN05513850,GSM2262777",10,hour post pupal formation,,,,,,,,,15/07/2026,"Dissected tissue was stored in Trizol reagent for pooling. RNA was extracted using the RNeasy kit (Qiagen) according to the manufacturer’s instructions, including an on-column DNase digestion (Qiagen). RNA concentrations were measured and 700ng RNA was diluted to 60ul total volume with H2O. RNA sequencing libraries were prepared with TruSeq Stran

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[:,'sex'] = ''
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [9]:
# making these variables because we use them again in the experiment file
my_protocol = 'TruSeq Stranded mRNA'
# full_length or 3'
my_protocolType = 'full_length'

library.loc[:,'protocol'] = my_protocol
library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'polyA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX2004106,SRP080993,Illumina HiSeq 2500,SRS1603673,UBERON:0000972,antenna,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262779,Antenna,48hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p40_2,"SAMN05513848,GSM2262779",48,hour post pupal formation,,,,,,,,,15/07/2026,"Dissected tissue was stored in Trizol reagent for pooling. RNA was extracted using the RNeasy kit (Qiagen) according to the manufacturer’s instructions, including an on-column DNase digestion (Qiagen). RNA concentrations were measured and 700ng RNA was diluted to 60ul total volume with H2O. RNA sequencing libraries were prepared with TruSeq Stranded mRNA Sample Prep Kit (Illumina) according to manufacturer’s instructions. For the RNA fragmentation step, 94˚C, 2min was used with the intention to obtain a median size ~185bp. PCR amplification was done with 15 cycles. A total of 24 multiplexed libraries (barcoded) were accessed for quality and combined together before separation into two identical pooled libraries, which were then subjected to cluster generation followed by Illumina 50bp paired-end sequencing by the UNC High-Throughput Sequencing Facility (HTSF).",,,,48hr APF pupal antenna,70,48hrs after pupal formation,,TRANSCRIPTOMIC,cDNA
1,SRX2004105,SRP080993,Illumina HiSeq 2500,SRS1603670,UBERON:0000972,antenna,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262778,Antenna,48hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p40_1,"SAMN05513849,GSM2262778",48,hour post pupal formation,,,,,,,,,15/07/2026,"Dissected tissue was stored in Trizol reagent for pooling. RNA was extracted using the RNeasy kit (Qiagen) according to the manufacturer’s instructions, including an on-column DNase digestion (Qiagen). RNA concentrations were measured and 700ng RNA was diluted to 60ul total volume with H2O. RNA sequencing libraries were prepared with TruSeq Stranded mRNA Sample Prep Kit (Illumina) according to manufacturer’s instructions. For the RNA fragmentation step, 94˚C, 2min was used with the intention to obtain a median size ~185bp. PCR amplification was done with 15 cycles. A total of 24 multiplexed libraries (barcoded) were accessed for quality and combined together before separation into two identical pooled libraries, which were then subjected to cluster generation followed by Illumina 50bp paired-end sequencing by the UNC High-Throughput Sequencing Facility (HTSF).",,,,48hr APF pupal antenna,70,48hrs after pupal formation,,TRANSCRIPTOMIC,cDNA
2,SRX2004104,SRP080993,Illumina HiSeq 2500,SRS1603668,UBERON:6001767,insect antennal disc,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262777,Antennal disc,10hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p8_2,"SAMN05513850,GSM2262777",10,hour post pupal formation,,,,,,,,,15/07/2026,"Dissected tissue was stored in Trizol reagent for pooling. RNA was extracted using the RNeasy kit (Qiagen) according to the manufacturer’s instructions, including an on-column DNase digestion (Qiagen). RNA concentrations were measured and 700ng RNA was diluted to 60ul total volume with H2O. RNA sequencing libraries were prepared with TruSeq Stran

#### globin, replicates

In [10]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [ ]:
#library.loc[:,'sampleAge_value'] = ''
#library.loc[:,'sampleAge_unit'] = ''

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [11]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2026-07-15'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection
0,SRX2004106,SRP080993,Illumina HiSeq 2500,SRS1603673,UBERON:0000972,antenna,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262779,Antenna,48hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p40_2,"SAMN05513848,GSM2262779",48,hour post pupal formation,,,,,,,,SAC,2026-07-15,"Dissected tissue was stored in Trizol reagent for pooling. RNA was extracted using the RNeasy kit (Qiagen) according to the manufacturer’s instructions, including an on-column DNase digestion (Qiagen). RNA concentrations were measured and 700ng RNA was diluted to 60ul total volume with H2O. RNA sequencing libraries were prepared with TruSeq Stranded mRNA Sample Prep Kit (Illumina) according to manufacturer’s instructions. For the RNA fragmentation step, 94˚C, 2min was used with the intention to obtain a median size ~185bp. PCR amplification was done with 15 cycles. A total of 24 multiplexed libraries (barcoded) were accessed for quality and combined together before separation into two identical pooled libraries, which were then subjected to cluster generation followed by Illumina 50bp paired-end sequencing by the UNC High-Throughput Sequencing Facility (HTSF).",,,,48hr APF pupal antenna,70,48hrs after pupal formation,,TRANSCRIPTOMIC,cDNA
1,SRX2004105,SRP080993,Illumina HiSeq 2500,SRS1603670,UBERON:0000972,antenna,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262778,Antenna,48hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p40_1,"SAMN05513849,GSM2262778",48,hour post pupal formation,,,,,,,,SAC,2026-07-15,"Dissected tissue was stored in Trizol reagent for pooling. RNA was extracted using the RNeasy kit (Qiagen) according to the manufacturer’s instructions, including an on-column DNase digestion (Qiagen). RNA concentrations were measured and 700ng RNA was diluted to 60ul total volume with H2O. RNA sequencing libraries were prepared with TruSeq Stranded mRNA Sample Prep Kit (Illumina) according to manufacturer’s instructions. For the RNA fragmentation step, 94˚C, 2min was used with the intention to obtain a median size ~185bp. PCR amplification was done with 15 cycles. A total of 24 multiplexed libraries (barcoded) were accessed for quality and combined together before separation into two identical pooled libraries, which were then subjected to cluster generation followed by Illumina 50bp paired-end sequencing by the UNC High-Throughput Sequencing Facility (HTSF).",,,,48hr APF pupal antenna,70,48hrs after pupal formation,,TRANSCRIPTOMIC,cDNA
2,SRX2004104,SRP080993,Illumina HiSeq 2500,SRS1603668,UBERON:6001767,insect antennal disc,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262777,Antennal disc,10hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p8_2,"SAMN05513850,GSM2262777",10,hour post pupal formation,,,,,,,,SAC,2026-07-15,"Dissected tissue was stored in Trizol reagent for pooling. RNA was extracted using the RNeasy kit (Qiagen) according to the manufacturer’s instructions, including an on-column DNase digestion (Qiagen). RNA concentrations were measured and 700ng RNA was diluted to 60ul total volume with H2O. RNA sequencing libraries were prepared with Tru

#### comments

In [12]:
library.loc[:,'comment'] = 'PMID: 28821769'

#### save complete file with correct columns

In [13]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX2004106,SRP080993,Illumina HiSeq 2500,SRS1603673,UBERON:0000972,antenna,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262779,Antenna,48hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p40_2,"SAMN05513848,GSM2262779",48,hour post pupal formation,,,,,PMID: 28821769,,,SAC,2026-07-15
1,SRX2004105,SRP080993,Illumina HiSeq 2500,SRS1603670,UBERON:0000972,antenna,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262778,Antenna,48hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p40_1,"SAMN05513849,GSM2262778",48,hour post pupal formation,,,,,PMID: 28821769,,,SAC,2026-07-15
2,SRX2004104,SRP080993,Illumina HiSeq 2500,SRS1603668,UBERON:6001767,insect antennal disc,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262777,Antennal disc,10hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p8_2,"SAMN05513850,GSM2262777",10,hour post pupal formation,,,,,PMID: 28821769,,,SAC,2026-07-15
3,SRX2004103,SRP080993,Illumina HiSeq 2500,SRS1603672,UBERON:6001767,insect antennal disc,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262776,Antennal disc,10hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p8_1,"SAMN05513851,GSM2262776",10,hour post pupal formation,,,,,PMID: 28821769,,,SAC,2026-07-15
4,SRX2004102,SRP080993,Illumina HiSeq 2500,SRS1603671,UBERON:6001767,insect antennal disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262775,Antennal disc,3rd instar larvae,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_3L_2,"SAMN05513852,GSM2262775",,,,,,,PMID: 28821769,,,SAC,2026-07-15
5,SRX2004101,SRP080993,Illumina HiSeq 2500,SRS1603667,UBERON:6001767,insect antennal disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262774,Antennal disc,3rd instar larvae,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_3L_1,"SAMN05513853,GSM2262774",,,,,,,PMID: 28821769,,,SAC,2026-07-15
6,SRX2004100,SRP080993,Illumina HiSeq 2500,SRS1603664,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262773,Antenna,Adult,perfect match,not documented,perfect match,mixed,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_AA_2,"SAMN05513854,GSM2262773",7-10,day post eclosion,,,,,PMID: 28821769,,,SAC,2026-07-15
7,SRX2004099,SRP080993,Illumina HiSeq 2500,SRS1603669,UBERON:0000972,antenna,UBERON:0000066,fully formed stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262772,Antenna,Adult,perfect match,not documented,perfect match,mixed,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_AA_1,"SAMN05513855,GSM2262772",7-10,day post eclosion,,,,,PMID: 28821769,,,SAC,2026-07-15
8,SRX2004098,SRP080993,Illumina HiSeq 2500,SRS1603663,UBERON:0000972,antenna,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSM2262771,Antenna,40hrs after pupal formation,perfect match,not documented,perfect match,NA,14021-0251.165,,7240,TruSeq Stranded mRNA,full_length,polyA,,,sim_p40_2,"SAMN05513856

### experiment annotations

In [14]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP080993,Developmental hotspots drive transcriptional variability and convergence in the Drosophila olfactory system,"The goal of this study was to identify the preponderance of developmental hotspots in the Drosophila olfactory system. Using RNASeq-derived transcriptome profiles of the developing and adult antennae for six different Drosophila species, we show that a few highly variable transcription factors may drive, likely in a combinatorial fashion, high variability in a few select olfactory receptor neuron (ORN) developmental lineages (i.e. developmental hotspots) in an otherwise mostly conserved background. Furthermore, the high variability of these few ORN lineages leads to a correspondingly high probability that they will recruited during the adaptation of the Drosophila olfactory system to specific ecological pressures, one potential example being the remarkably convergent antennal transcriptome profiles in two Drosophila species that have independently evolved the ability to specialize on specific host plants. Overall design: mRNA profiles from antenna and antennal discs across 4 developmental time points for 6 Drosophila species",SRA,,,,TruSeq Stranded mRNA,full_length,GSE85239,PRJNA337934,"28644712, 28821769",,"10.1038/s41598-017-08563-0,10.1080/19336934.2017.1344374",,


#### experiment and protocol details

In [15]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

40

In [16]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'total'
# Bgee 1K
experiment.loc[:,'projectTags'] = 'Bgee 1K' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
experiment.loc[:,'protocol'] = my_protocol
experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP080993,Developmental hotspots drive transcriptional variability and convergence in the Drosophila olfactory system,"The goal of this study was to identify the preponderance of developmental hotspots in the Drosophila olfactory system. Using RNASeq-derived transcriptome profiles of the developing and adult antennae for six different Drosophila species, we show that a few highly variable transcription factors may drive, likely in a combinatorial fashion, high variability in a few select olfactory receptor neuron (ORN) developmental lineages (i.e. developmental hotspots) in an otherwise mostly conserved background. Furthermore, the high variability of these few ORN lineages leads to a correspondingly high probability that they will recruited during the adaptation of the Drosophila olfactory system to specific ecological pressures, one potential example being the remarkably convergent antennal transcriptome profiles in two Drosophila species that have independently evolved the ability to specialize on specific host plants. Overall design: mRNA profiles from antenna and antennal discs across 4 developmental time points for 6 Drosophila species",SRA,total,Bgee 1K,40,TruSeq Stranded mRNA,full_length,GSE85239,PRJNA337934,"28644712, 28821769",,"10.1038/s41598-017-08563-0,10.1080/19336934.2017.1344374",,


#### paper and xrefs

In [17]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '28821769'
experiment.loc[:,'reference_url'] = 'https://pmc.ncbi.nlm.nih.gov/articles/PMC5562767/'
experiment.loc[:,'DOI'] = '10.1038/s41598-017-08563-0'
experiment.loc[:,'xrefs'] = '28644712[PMID]'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP080993,Developmental hotspots drive transcriptional variability and convergence in the Drosophila olfactory system,"The goal of this study was to identify the preponderance of developmental hotspots in the Drosophila olfactory system. Using RNASeq-derived transcriptome profiles of the developing and adult antennae for six different Drosophila species, we show that a few highly variable transcription factors may drive, likely in a combinatorial fashion, high variability in a few select olfactory receptor neuron (ORN) developmental lineages (i.e. developmental hotspots) in an otherwise mostly conserved background. Furthermore, the high variability of these few ORN lineages leads to a correspondingly high probability that they will recruited during the adaptation of the Drosophila olfactory system to specific ecological pressures, one potential example being the remarkably convergent antennal transcriptome profiles in two Drosophila species that have independently evolved the ability to specialize on specific host plants. Overall design: mRNA profiles from antenna and antennal discs across 4 developmental time points for 6 Drosophila species",SRA,total,Bgee 1K,40,TruSeq Stranded mRNA,full_length,GSE85239,PRJNA337934,28821769,https://pmc.ncbi.nlm.nih.gov/articles/PMC5562767/,10.1038/s41598-017-08563-0,28644712[PMID],


#### comments

In [ ]:
#experiment.loc[:,'comment'] = ''

display_df(experiment)

#### save complete file

In [18]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [19]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

In [20]:
! python3 $path_to_v_script --bulk-exp $experiment_to_add_path --bulk-lib $library_to_add_path --rules-dir $path_to_rules --out $val_output --strict

Total issues: 30
Errors: 30
Warnings: 0
Top codes:
  - BAD_VALUE: 30


#### check columns match

In [23]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [24]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
68014,SRX6772288,SRP219656,Illumina HiSeq 2500,SRS5319587,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,stage 2 embryo,stage 2,perfect match,not documented,perfect match,NA,(D. simulans x D. sechellia) x D. simulans,,1489344,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans x D. sechellia F1 backcross, stage...","SAMN12659518,GSM4054464",,,,,,,PMID: 32928902,,,SAC,2026-07-15
68015,SRX6772287,SRP219656,Illumina HiSeq 2500,SRS5319586,UBERON:0007010,cleaving embryo,UBERON:0000107,cleavage stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,stage 2 embryo,stage 2,perfect match,not documented,perfect match,NA,(D. simulans x D. sechellia) x D. simulans,,1489344,TruSeq Stranded mRNA,full_length,polyA,,,"D. simulans x D. sechellia F1 backcross, stage...","SAMN12659484,GSM4054463",,,,,,,PMID: 32928902,,,SAC,2026-07-15
68016,SRX2004106,SRP080993,Illumina HiSeq 2500,SRS1603673,UBERON:0000972,antenna,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Antenna,48hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p40_2,"SAMN05513848,GSM2262779",48,hour post pupal formation,,,,,PMID: 28821769,,,SAC,2026-07-15
68017,SRX2004105,SRP080993,Illumina HiSeq 2500,SRS1603670,UBERON:0000972,antenna,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Antenna,48hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p40_1,"SAMN05513849,GSM2262778",48,hour post pupal formation,,,,,PMID: 28821769,,,SAC,2026-07-15
68018,SRX2004104,SRP080993,Illumina HiSeq 2500,SRS1603668,UBERON:6001767,insect antennal disc,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Antennal disc,10hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p8_2,"SAMN05513850,GSM2262777",10,hour post pupal formation,,,,,PMID: 28821769,,,SAC,2026-07-15
68019,SRX2004103,SRP080993,Illumina HiSeq 2500,SRS1603672,UBERON:6001767,insect antennal disc,UBERON:0000070,pupal stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Antennal disc,10hrs after pupal formation,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_p8_1,"SAMN05513851,GSM2262776",10,hour post pupal formation,,,,,PMID: 28821769,,,SAC,2026-07-15
68020,SRX2004102,SRP080993,Illumina HiSeq 2500,SRS1603671,UBERON:6001767,insect antennal disc,UBERON:8000002,third instar larva stage,http://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?...,Antennal disc,3rd instar larvae,perfect match,not documented,perfect match,NA,15010-1051.00,,7244,TruSeq Stranded mRNA,full_length,polyA,,,vir_3L_2,"SAMN05513852,GSM2262775",,,,,,,PMID: 28821769,,,SAC,2026-07-15


In [25]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
1289,ERP137517,RNAseq for European Seabass tissues,The European project AQUA-FAANG aims to improv...,SRA,total,AQUA-FAANG,84,NEBNext Ultra Directional RNA Library Prep Kit,full_length,,PRJEB52776,,https://projects.ensembl.org/aqua-faang/,,,
1290,SRP219656,Evolved differences in cis and trans regulatio...,"During embryogenesis in animals, initial devel...",SRA,total,Bgee 1K,63,TruSeq Stranded mRNA,full_length,GSE136646,PRJNA562920,32928902,https://pmc.ncbi.nlm.nih.gov/articles/PMC7648588/,10.1534/genetics.120.303626,,
1291,SRP080993,Developmental hotspots drive transcriptional v...,The goal of this study was to identify the pre...,SRA,total,Bgee 1K,40,TruSeq Stranded mRNA,full_length,GSE85239,PRJNA337934,28821769,https://pmc.ncbi.nlm.nih.gov/articles/PMC5562767/,10.1038/s41598-017-08563-0,28644712[PMID],


### add annotations to git

In [26]:
! git pull

Already up to date.


In [27]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [28]:
! git add $git_experiment_path $git_library_path

In [29]:
! git commit -m $commit_message_exp

[develop 3db653b] adding annotated bulk experiment SRP080993
 2 files changed, 41 insertions(+)


In [30]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 3.82 KiB | 978.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0 (from 0)
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git/
   97ab885..3db653b  develop -> develop


### add annotation folder and script to git

In [ ]:
! git pull

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push